# Final Individual Project (Cycle 4): Regression Analysis
**Student Name :** 彭佳淳  
**Student ID :** 113370216 

## Step 1: Introduction & Research Questions

### 青春期的隱形牢籠：青少年親密關係暴力與性創傷對自殺意念的加乘效應
### The Compounding Effect of Intimate Partner Violence and Sexual Trauma on Adolescent Suicidal Ideation

---
### 一、 研究主題與多變數架構 (Research Topic & Multi-Variable Framework)  
  本研究基於美國 CDC 的 `YRBS_2007.csv` 數據集，探討「青少年遭受親密關係暴力與強迫性行為等創傷，是否會顯著增加其考慮自殺的風險（機率）」。  
  為了符合期末專題之深度並展現完整的統計工作流（Statistical Workflow），本專案將反應變數（Response Variable）鎖定為二元變數 `ConsideredSuicide`（是否曾考慮自殺），並透過特徵工程建立複合創傷指標，進行**羅吉斯迴歸分析（Logistic Regression）**：
  
  1. **核心預測變數（Predictor Variables / 創傷維度）**
     * **親密關係暴力**：`BoyfriendGirlfriendPhysicallyHurt` (遭受男女朋友身體傷害)
     * **性創傷**：`ForcedSexualIntercourse` (曾被強迫發生性行為)
     * **複合創傷指標（Feature Engineering）**：將上述兩者結合，分為「無創傷 (No Trauma)」、「單一創傷 (Either Trauma)」與「雙重創傷 (Both Traumas)」。
  2. **控制變數（Control Variables / 干擾因素）**
     * **生理與人口統計**：女學生（Female） vs 男學生（Male），以控制性別在心理健康上的基準差異，並藉此探討潛在的性別脆弱性落差（Gender Gap in Vulnerability）。

* **English Description**:  
  This study is based on the CDC's `YRBS_2007.csv` dataset, strictly focusing on examining whether exposure to intimate partner violence and forced sexual intercourse significantly increases the probability of suicidal ideation among adolescents. To demonstrate a complete statistical workflow using Logistic Regression, our binary response variable is `ConsideredSuicide`. We engineer a composite trauma index (No Trauma, Either Trauma, Both Traumas) as the primary predictor, while controlling for biological sex to isolate the direct impact of trauma and explore gender-based psychological vulnerabilities.

---

### 二、 研究手法與資料清洗規則 (Research Methodology & Recoding Rules)
* **中文說明**：  
  本研究將嚴格定義變數並進行重編碼，以確保羅吉斯迴歸模型的精確性與解釋性（定義 $1 = $ 是/有，$0 = $ 否/無）：
  * **反應變數**：`ConsideredSuicide` $\rightarrow$ 原始代碼 1 轉為 `1`（曾考慮自殺）；代碼 2 轉為 `0`（未考慮）。
  * **創傷分組**：`BoyfriendGirlfriendPhysicallyHurt` 與 `ForcedSexualIntercourse` $\rightarrow$ 代碼 1 轉為 `1`（有遭遇）；代碼 2 轉為 `0`（無遭遇）。
  * **性別分組**：`WhatIsYourSex` $\rightarrow$ 代碼 1 轉為 `Female` ；代碼 2 轉為 `Male`。
  
  所有包含缺失值（NA）的樣本將採用完全個案剔除法（Listwise Deletion）排除，確保模型推論建立在完全乾淨的有效樣本上。

---

### 三、 統計分析方法與假設驗證 (Statistical Methods & Assumptions)
* **中文說明**：  
  由於本研究的反應變數為**二元類別變數**，我們將採用教授核准之 **羅吉斯迴歸模型（Logistic Regression）** 進行推論統計：
  1. **描述性統計與視覺化**：繪製不同創傷維度下的自殺意念比例（Bar Charts）與性別交互作用熱圖（Heatmap）。
  2. **模型建立**：建立多元羅吉斯迴歸模型，將 `ConsideredSuicide` 作為依變數（Y），創傷指標與性別變數作為自變數（X）。
  3. **勝算比（Odds Ratio, OR）解析**：計算並解釋自變數的勝算比與 95% 信賴區間，量化創傷疊加對自殺風險的具體倍數影響。
  4. **因果關係規避聲明**：由於本數據為觀察性調查，所有結論僅闡述「統計相關性（Statistical Association）」，**絕不推導出絕對的因果關係**。
---

## 1-1 Setup & Data Loading

> **Methodological Protocol / 方法學協議**：
> This component programmatically establishes a standardized, decoupled workspace directory under an absolute project root. This ensures strict configuration management and prevents pipeline failures caused by runtime path drifts.
> 
> 本步驟透過程式自動在專案根目錄下建置標準化、完全解耦的分層資料夾。此架構確保了嚴格的配置管理，徹底防止分析管線因本機端執行路徑漂移而發生讀檔錯誤。

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

# 設定繪圖風格與高解析度
sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 120

# 1. 自動偵測專案根目錄 (根據您的要求設定為 cycle4)
base_project_dir = r"C:\Users\88690\Desktop\cycle4"

# 2. 定義各個標準分層資料夾的絕對路徑
data_raw_dir = os.path.join(base_project_dir, 'data', 'raw')
data_processed_dir = os.path.join(base_project_dir, 'data', 'processed')
outputs_figures_dir = os.path.join(base_project_dir, 'outputs', 'figures')
outputs_tables_dir = os.path.join(base_project_dir, 'outputs', 'tables')

# 3. 如果資料夾不存在，則自動建立
directories = [data_raw_dir, data_processed_dir, outputs_figures_dir, outputs_tables_dir]
for directory in directories:
    if not os.path.exists(directory):
        os.makedirs(directory)

# 4. 指定原始資料 YRBS_2007.csv 的精確絕對路徑
raw_data_path = os.path.join(data_raw_dir, 'YRBS_2007.csv')

# 5. 執行檔案讀取
try:
    df = pd.read_csv(raw_data_path)
    print(f"【成功】原始數據集已從絕對路徑成功載入！")
    print(f"原始資料維度 (Raw Data Shape): {df.shape[0]} 列, {df.shape[1]} 欄")
except FileNotFoundError:
    print(f"❌【錯誤】在路徑 {raw_data_path} 依然找不到檔案。")
    print("請確保您的 YRBS_2007.csv 檔案已放置於 C:\\Users\\88690\\Desktop\\cycle4\\data\\raw\\ 資料夾中。")

【成功】原始數據集已從絕對路徑成功載入！
原始資料維度 (Raw Data Shape): 14041 列, 103 欄


資料集與核心分析環境已成功初始化。透過絕對路徑定位，原始數據載入正確，且專案標準資料夾結構（`data/`, `outputs/`）已建立完畢。接下來我們將進行核心變數的二元重編碼與深度清洗。

The dataset and the core analytical environment have been successfully initialized. Using absolute paths, the raw data loaded correctly, and the standard directory structure has been ensured. Next, we will proceed to binary recoding and advanced data cleaning.

## 1.2 數據準備與變數重編碼
## Data Preparation and Recoding

> **Methodological Protocol / 方法學協議**：
> This module programmatically binarizes core YRBS vectors and executes a strict listwise deletion protocol (`.dropna()`). By standardizing raw questionnaire codes into strictly bounded biological scales and integer types (`int`), this pipeline synchronizes the analytical baseline and exports a pristine dataset (`yrbs_cleaned_trauma.csv`) for rigorous downstream statistical inference.
> 
> 本模組對 YRBS 核心向量進行二元化轉換，並透過完全個案剔除法（Listwise Deletion）過濾缺失值與非目標雜訊。管線將問卷原始代碼標準化為具備嚴格數學邊界的二元指標與整數型態，旨在同步樣本分析基準，並導出高純淨度數據集供下游推論統計使用。

In [5]:
# ======================================================================
# 階段 A：獨立執行變數重編碼與特徵工程 (Data Recoding & Feature Engineering)
# ======================================================================

# 1. 挑選本研究所需的核心分析原始欄位
target_cols = [
    'WhatIsYourSex', 
    'BoyfriendGirlfriendPhysicallyHurt', 
    'ForcedSexualIntercourse', 
    'ConsideredSuicide'
]
df_sub = df[target_cols].copy()

# 2. 核心變數二元化 (1 -> 1: 是/有, 2 -> 0: 否/無)
# 反應變數：是否考慮自殺
df_sub['Suicide_Binary'] = df_sub['ConsideredSuicide'].map({1: 1, 2: 0})

# 創傷變數
df_sub['PartnerViolence_Binary'] = df_sub['BoyfriendGirlfriendPhysicallyHurt'].map({1: 1, 2: 0})
df_sub['ForcedSex_Binary'] = df_sub['ForcedSexualIntercourse'].map({1: 1, 2: 0})

# 3. 嚴格對齊 YRBS 官方定義 (1 -> Female 女性, 2 -> Male 男性)
df_sub['Sex_Label'] = df_sub['WhatIsYourSex'].map({1: 'Female', 2: 'Male'})

# 4. 特徵工程 (Feature Engineering)：建立「複合創傷 (Trauma_Level)」變數
# 將兩項創傷分數相加 (0=無創傷, 1=單一創傷, 2=雙重創傷)
df_sub['Trauma_Score'] = df_sub['PartnerViolence_Binary'] + df_sub['ForcedSex_Binary']

def classify_trauma(score):
    if score == 2.0:
        return 'Both Traumas'
    elif score == 1.0:
        return 'Either Trauma'
    elif score == 0.0:
        return 'No Trauma'
    else:
        return np.nan

df_sub['Trauma_Level'] = df_sub['Trauma_Score'].apply(classify_trauma)

print("\n重編碼後的資料前五筆預覽：")
print(df_sub[['Sex_Label', 'Suicide_Binary', 'PartnerViolence_Binary', 'ForcedSex_Binary', 'Trauma_Level']].head())


重編碼後的資料前五筆預覽：
  Sex_Label  Suicide_Binary  PartnerViolence_Binary  ForcedSex_Binary  \
0      Male             0.0                     1.0               1.0   
1      Male             0.0                     0.0               0.0   
2      Male             0.0                     0.0               0.0   
3    Female             1.0                     1.0               1.0   
4    Female             NaN                     1.0               1.0   

   Trauma_Level  
0  Both Traumas  
1     No Trauma  
2     No Trauma  
3  Both Traumas  
4  Both Traumas  


In [6]:
# ======================================================================
# 階段 B：缺失數據與雜訊值剔除及學術檔案導出 (Data Cleaning & Export)
# ======================================================================
initial_count = df_sub.shape[0]

# 1. 定義重編碼後我們必須保留的乾淨新欄位
clean_cols = [
    'Suicide_Binary', 
    'Sex_Label', 
    'PartnerViolence_Binary', 
    'ForcedSex_Binary', 
    'Trauma_Level'
]

# 2. 執行完全個案剔除法 (.dropna() 會自動過濾掉任何含有 NaN 的不完整資料列)
df_clean = df_sub[clean_cols].dropna().reset_index(drop=True)

# 3. 確保二元變數以標準學術計量整數型態（int）儲存
df_clean['Suicide_Binary'] = df_clean['Suicide_Binary'].astype(int)
df_clean['PartnerViolence_Binary'] = df_clean['PartnerViolence_Binary'].astype(int)
df_clean['ForcedSex_Binary'] = df_clean['ForcedSex_Binary'].astype(int)

# 4. 導出至 processed 資料夾 (依據要求存入 C:\Users\88690\Desktop\cycle4\data\processed)
processed_csv_path = os.path.join(data_processed_dir, 'yrbs_cleaned_data.csv')
df_clean.to_csv(processed_csv_path, index=False)

# 5. 印出學術審報日誌，並「即時動態驗證」創傷層級與自殺比例的關係
print("============ ✅ Data Cleaning Protocol Executed ============")
print("缺失值與無效代碼已完全剔除（Listwise Deletion 已完成）。")
print(f"  - 原始基礎欄位總數: {initial_count} 筆")
print(f"  - 最終純淨有效樣本數: {df_clean.shape[0]} 筆")

print("\n📊 驗證導出的真實數據：不同創傷層級的自殺意念盛行率")
verification = df_clean.groupby('Trauma_Level', observed=False)['Suicide_Binary'].mean() * 100
# 排序顯示以符合邏輯遞增
for level in ['No Trauma', 'Either Trauma', 'Both Traumas']:
    if level in verification:
        print(f"  - {level} 實際考慮自殺比例: {verification[level]:.2f}%")

print(f"\n👉 乾淨數據集已成功儲存至:\n   👉 {processed_csv_path}")

============ ✅ Data Cleaning Protocol Executed ============
缺失值與無效代碼已完全剔除（Listwise Deletion 已完成）。
  - 原始基礎欄位總數: 14041 筆
  - 最終純淨有效樣本數: 13737 筆

📊 驗證導出的真實數據：不同創傷層級的自殺意念盛行率
  - No Trauma 實際考慮自殺比例: 11.96%
  - Either Trauma 實際考慮自殺比例: 26.97%
  - Both Traumas 實際考慮自殺比例: 47.59%

👉 乾淨數據集已成功儲存至:
   👉 C:\Users\88690\Desktop\cycle4\data\processed\yrbs_cleaned_data.csv


數據準備與變數重編碼已成功落實。所有非目標代碼與缺失值均已被排除，獲得了結構純淨的有效樣本數據，為後續的敘述統計與 Logistic Regression 推論統計奠定了正確的基礎。

Data preparation and binary recoding have been successfully completed. All non-target noise codes and missing observations have been eliminated, establishing a solid, error-free foundation for subsequent descriptive and inferential analysis using Logistic Regression.